# Problem Statement

#### Employee Attrition Prediction using Logistic Regression

- A company is experiencing employee turnover and wants to proactively identify employees who are likely to leave the organization.

- The goal of this project is to build a Logistic Regression classification model that predicts whether an employee will:

- Leave the company (Attrition = 1)
- Stay in the company (Attrition = 0)

using employee-related factors such as:

- age
- monthly income
- years at company
- job satisfaction
- work-life balance
- overtime
- department
- job role

### Business Objective

The company wants to:
- reduce employee turnover
- improve employee retention
- identify high-risk employees early
- support HR decision-making using data


### Machine Learning Objective

Develop a supervised classification model using Logistic Regression to predict employee attrition based on historical employee data.


### Target Variable

###### Attrition

Where:
- 1 = Employee leaves
- 0 = Employee stays


### Success Metrics

The model will be evaluated using:
- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC
- Confusion Matrix


### Expected Outcome

The final model should help HR departments:
- identify employees at risk of leaving
- improve retention strategies
- reduce recruitment and training costs



### Load dataset

In [3]:
# Import libraries for data handling
import pandas as pd
import numpy as np

# Import libraries for visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Evaluation metrics
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

# Preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

In [4]:
# Load the dataset
df = pd.read_csv("employee_attrition.csv")
df.head()

,EmployeeID,Age,MonthlyIncome,YearsAtCompany,JobSatisfaction,WorkLifeBalance,OverTime,Department,JobRole,Attrition
0,20001,53,2694.0,4,3,1,No,HR,Software Engineer,0
1,20002,41,2273.0,7,4,1,No,Operations,Software Engineer,0
2,20003,43,2317.0,10,3,2,No,Tech,Data Analyst,0
3,20004,51,2549.0,2,4,4,No,Tech,HR Specialist,0
4,20005,40,3861.0,10,2,2,No,Tech,Software Engineer,0


### Understand structure and Data Cleaning

In [5]:
# understand the data
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   EmployeeID       600 non-null    int64
 1   Age              600 non-null    int64
 2   MonthlyIncome    600 non-null    str  
 3   YearsAtCompany   600 non-null    int64
 4   JobSatisfaction  600 non-null    int64
 5   WorkLifeBalance  600 non-null    int64
 6   OverTime         600 non-null    str  
 7   Department       600 non-null    str  
 8   JobRole          600 non-null    str  
 9   Attrition        600 non-null    int64
dtypes: int64(6), str(4)
memory usage: 47.0 KB


In [6]:
# Value Counts
df["Attrition"].value_counts()

Attrition
0    469
1    131
Name: count, dtype: int64

#### We just discovered

1. MonthlyIncome -   str
2. OverTime      -   str
3. Department    -   str
4. JobRole       -   str
- These are currently strings, not proper numeric/categorical types.
- The 

In [7]:
# change the MonthlyIncome data type to numeric
df["MonthlyIncome"] = pd.to_numeric(df["MonthlyIncome"], errors="coerce")
df['MonthlyIncome'].dtype


dtype('float64')

In [8]:
# binary encoding for the OverTime column

df["OverTime"] = df["OverTime"].map({"No": 0, "Yes": 1})
df["OverTime"].head()


0    0
1    0
2    0
3    0
4    0
Name: OverTime, dtype: int64

In [9]:
# check unique values in JobRole column

df['JobRole'].unique()



# Replace '?' with NaN in JobRole column
df['JobRole'] = df['JobRole'].replace('?', np.nan)

In [10]:

# Check if ? is present in other clumns
(df == '?').sum()

EmployeeID         0
Age                0
MonthlyIncome      0
YearsAtCompany     0
JobSatisfaction    0
WorkLifeBalance    0
OverTime           0
Department         0
JobRole            0
Attrition          0
dtype: int64

In [11]:
# check for missing values
df.isnull().sum()

EmployeeID          0
Age                 0
MonthlyIncome      18
YearsAtCompany      0
JobSatisfaction     0
WorkLifeBalance     0
OverTime            0
Department          0
JobRole            12
Attrition           0
dtype: int64

In [12]:
# Fill missing values in MonthlyIncome with the median
df["MonthlyIncome"] = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())

# Fill missing values in JobRole with the mode
df["JobRole"] = df["JobRole"].fillna(df["JobRole"].mode()[0])


In [13]:
df.isnull().sum()

EmployeeID         0
Age                0
MonthlyIncome      0
YearsAtCompany     0
JobSatisfaction    0
WorkLifeBalance    0
OverTime           0
Department         0
JobRole            0
Attrition          0
dtype: int64

### Preprocess features

In [14]:
# Hot encode categorical columns
df = pd.get_dummies(df, columns=["Department", "JobRole"], drop_first=True)

In [15]:
df.head()

,EmployeeID,Age,MonthlyIncome,YearsAtCompany,JobSatisfaction,WorkLifeBalance,OverTime,Attrition,Department_Operations,Department_Sales,Department_Tech,JobRole_HR Specialist,JobRole_Operations Associate,JobRole_Sales Rep,JobRole_Software Engineer
0,20001,53,2694.0,4,3,1,0,0,False,False,False,False,False,False,True
1,20002,41,2273.0,7,4,1,0,0,True,False,False,False,False,False,True
2,20003,43,2317.0,10,3,2,0,0,False,False,True,False,False,False,False
3,20004,51,2549.0,2,4,4,0,0,False,False,True,True,False,False,False
4,20005,40,3861.0,10,2,2,0,0,False,False,True,False,False,False,True


In [16]:
df = df.astype(int)

In [17]:
df.head()

,EmployeeID,Age,MonthlyIncome,YearsAtCompany,JobSatisfaction,WorkLifeBalance,OverTime,Attrition,Department_Operations,Department_Sales,Department_Tech,JobRole_HR Specialist,JobRole_Operations Associate,JobRole_Sales Rep,JobRole_Software Engineer
0,20001,53,2694,4,3,1,0,0,0,0,0,0,0,0,1
1,20002,41,2273,7,4,1,0,0,1,0,0,0,0,0,1
2,20003,43,2317,10,3,2,0,0,0,0,1,0,0,0,0
3,20004,51,2549,2,4,4,0,0,0,0,1,1,0,0,0
4,20005,40,3861,10,2,2,0,0,0,0,1,0,0,0,1


### Train model

In [18]:
# Spliting the data into features and target variable
X = df.drop("Attrition", axis=1)
y = df["Attrition"]


##### Feature Scaling (IMPORTANT)

Logistic Regression performs better when numeric features are scaled.

We scale ONLY X (not y).

In [19]:
# Scale the features using MinMaxScaler

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

In [20]:
# Split the data into features and target variable
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)    


In [21]:
# Train the logistic regression model

model = LogisticRegression()    
model.fit(X_train, y_train)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [22]:
# Make predictions
y_pred = model.predict(X_test)
y_pred[:10]  # Display the first 10 predictions

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [23]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred))


Confusion Matrix:
[[88  0]
 [32  0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.73      1.00      0.85        88
           1       0.00      0.00      0.00        32

    accuracy                           0.73       120
   macro avg       0.37      0.50      0.42       120
weighted avg       0.54      0.73      0.62       120


Accuracy Score:
0.7333333333333333


c:\Users\P15s\OneDrive\Documents\Logistic Regression Project\ml_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\P15s\OneDrive\Documents\Logistic Regression Project\ml_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\P15s\OneDrive\Documents\Logistic Regression Project\ml_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to

#### Confusion Matrix Interpretation

- 88 employees were correctly predicted as “stay (0)”
- 0 employees were incorrectly predicted as “leave” (good for class 0)
- 32 employees actually left (1), but model missed ALL of them
- Model predicted 0 for every single person

##### Main problem

The model is not detecting employees who leave at all.

It only predicts:

“stay” for everyone

#### Classification Report (simple meaning)

Class 0 (Stay employees)
Precision: 0.73
→ When model says “stay”, it is right 73% of the time
Recall: 1.00
→ It correctly finds ALL stay employees
F1-score: 0.85
→ Overall good performance for class 0

Class 1 (Employees who leave)
Precision: 0.00
→ When model predicts “leave”, it is always wrong (but it never predicts leave anyway)
Recall: 0.00
→ Model fails to detect ANY employee who leaves
F1-score: 0.00
→ Completely failed class

Accuracy = 0.73
73% accuracy looks good
BUT it is misleading

👉 Why?
Because model just predicts “0” every time

##### Simple summary

👉 Model is good at predicting “who stays”
👉 Model is terrible at predicting “who leaves”



# Train improved model

In [24]:
# Improve the modelbalancing the class weight
model1 = LogisticRegression(class_weight="balanced")
model1.fit(X_train, y_train)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [25]:
# Predict again with the balanced model
y_pred_balanced = model1.predict(X_test)
y_pred_balanced[:10]  # Display the first 10 predictions


array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0])

In [26]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

[[88  0]
 [32  0]]
              precision    recall  f1-score   support

           0       0.73      1.00      0.85        88
           1       0.00      0.00      0.00        32

    accuracy                           0.73       120
   macro avg       0.37      0.50      0.42       120
weighted avg       0.54      0.73      0.62       120

Accuracy: 0.7333333333333333


c:\Users\P15s\OneDrive\Documents\Logistic Regression Project\ml_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\P15s\OneDrive\Documents\Logistic Regression Project\ml_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\P15s\OneDrive\Documents\Logistic Regression Project\ml_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to

## Recommendation (Simple)

- Logistic Regression model is not performing well for predicting employees who leave.
- It is only predicting “stay” for almost everyone.
- This means the model is not capturing the patterns for attrition properly.
- Even after fixing data, encoding, scaling, and balancing, the model still fails.
- This suggests the relationship in the data is too complex for Logistic Regression.
- The dataset likely needs a stronger model to detect non-linear patterns.

## What you should do next

Try a better model like Random Forest or Decision Tree
These models handle complex patterns better
They do not assume linear relationships like Logistic Regression

### Lets Try Random Forest 



In [29]:
# Importing the library for Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier


In [30]:
# Create and train the Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)



,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [31]:
# Let us make predictions with the Random Forest model
y_pred_rf = rf_model.predict(X_test)    
y_pred_rf[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [33]:
# Evaluate the balanced model

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred_rf))

Confusion Matrix:
[[86  2]
 [31  1]]

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.98      0.84        88
           1       0.33      0.03      0.06        32

    accuracy                           0.72       120
   macro avg       0.53      0.50      0.45       120
weighted avg       0.63      0.72      0.63       120


Accuracy Score:
0.725


##### Confusion Matrix
Meaning:
- 86 employees correctly predicted as “stay”
- 2 employees incorrectly predicted as “leave”
- 31 employees who actually left were missed
- 1 employee who left was correctly detected

🧠 What improved vs Logistic Regression?
✔ Before (Logistic Regression)
predicted ONLY 0
detected 0 employees who leave ❌
✔ Now (Random Forest)
detects 1 employee who leaves
makes some predictions for class 1
slightly better balance

##### Classification Report (simple explanation)

- Class 0 (Stay)
- Precision: 0.74 → good
- Recall: 0.98 → very good
- F1-score: 0.84 → strong

👉 Model is very good at predicting employees who stay

##### Next BEST step 

Now we fix the REAL issue:

👉 SMOTE + Random Forest

This will:

- balance classes properly
- force model to learn class 1
- improve recall significantly

In [36]:
# import SMOTE for handling class imbalance
from imblearn.over_sampling import SMOTE


In [37]:
# Apply SMOTE to the training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)


In [38]:
# Train the Random Forest Classifier on the SMOTE data
rf_model_smote = RandomForestClassifier(random_state=42)
rf_model_smote.fit(X_train_smote, y_train_smote)    


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [ ]:
# Predict with the SMOTE-trained model

y_pred_smote = rf_model_smote.predict(X_test)
y_pred_smote[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [40]:
# Evaluate the balanced model

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_smote))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_smote))

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred_smote))


Confusion Matrix:
[[82  6]
 [29  3]]

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.93      0.82        88
           1       0.33      0.09      0.15        32

    accuracy                           0.71       120
   macro avg       0.54      0.51      0.49       120
weighted avg       0.63      0.71      0.64       120


Accuracy Score:
0.7083333333333334


##### SMOTE Model Results (Simple Interpretation)

🟢 Confusion Matrix
- 82 employees correctly predicted as stay (0)
- 6 employees wrongly predicted as leave (1)
- 29 employees who actually left were missed
- 3 employees who left were correctly detected

##### What improved?

✔ Compared to previous models:

- It now detects 3 employees who leave (before it was 0 or 1)
- It is slightly more balanced than before
- It is no longer completely ignoring class 1

##### What is still a problem?

🔴 Class 1 (Leave) is still weak:

- Precision = 0.33 → when it predicts leave, it's often wrong
- Recall = 0.09 → it still misses most employees who leave
- F1-score = 0.15 → still very low

👉 This is the biggest weakness

##### Overall performance

- Accuracy: 70.8%
- Looks similar to before
- BUT still misleading because class 1 is poorly detected



# Feature Engineering. to improve more

In [41]:
# Create a new feature: income per year at company
# This shows how much income an employee has relative to their experience in the company
# We divide MonthlyIncome by (YearsAtCompany + 1) to avoid division by zero

df["IncomePerYear"] = df["MonthlyIncome"] / (df["YearsAtCompany"] + 1)

In [42]:
# Create a new feature: stress level
# This shows how stressed an employee might be based on satisfaction and work-life balance
# We reverse the scores because low satisfaction and poor balance increase stress risk

df["Stress_Level"] = (5 - df["JobSatisfaction"]) + (5 - df["WorkLifeBalance"])

##### Prepare Final Feature Set

In [43]:
# Create final dataset for modeling
# We remove raw identifiers and keep only useful numerical + encoded features

X = df.drop(["Attrition", "EmployeeID", "OverTime"], axis=1)
y = df["Attrition"]

In [44]:
# let us split the data again with the new features
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create improved Random Forest model
# This model will learn from engineered features like Stress_Level and IncomePerYear

rf_model_improved = RandomForestClassifier(random_state=42)

rf_model_improved.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [45]:
# Predict on test data using improved model
y_pred_improved = rf_model_improved.predict(X_test)

# Show first 10 predictions
y_pred_improved[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

#### Lets us evaluate

In [46]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_improved))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_improved))

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred_improved))

Confusion Matrix:
[[87  1]
 [32  0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.99      0.84        88
           1       0.00      0.00      0.00        32

    accuracy                           0.72       120
   macro avg       0.37      0.49      0.42       120
weighted avg       0.54      0.72      0.62       120


Accuracy Score:
0.725


# Conclusion

In this project, we built a machine learning model to predict employee attrition using HR data.

We started with data cleaning, where missing values were handled, data types were corrected, and categorical variables were encoded. We then performed feature engineering by creating new features such as IncomePerYear, OverTime_Risk, and Stress_Level to improve model understanding.

We trained multiple models including Logistic Regression and Random Forest. We also applied SMOTE to handle class imbalance in the dataset.

From the results, we observed that:
- Logistic Regression struggled to correctly identify employees who leave.
- Random Forest performed slightly better but still missed most attrition cases.
- SMOTE improved class balance but did not significantly improve final prediction of class 1 (attrition).

Overall, the models showed that predicting employee attrition is challenging using the available features, as the dataset has weak predictive signals and strong class imbalance.

Despite this, the project successfully demonstrated a complete machine learning pipeline including data cleaning, feature engineering, model training, and evaluation.

Future improvements could include using more advanced models such as XGBoost, adding richer HR features, and further feature engineering to improve predictive performance.